# 第 2 章　工具链：Git、Marp、Quarto 与 Stata×Python

> **T1 共三章**
> - 第 1 章：Python 环境配置
> - **第 2 章（本章）**：工具链
> - 第 3 章：金融数据 API 获取

::: {.callout-note}
## 本章要点

1. **Git + GitHub**：用版本控制管理代码，让分析可追溯、可分享
2. **Marp**：用 Markdown 写幻灯片，10 分钟出一份专业 slides
3. **Quarto**：把 Jupyter Notebook 变成网页报告或电子书
4. **Stata × Python**：在同一个 `.ipynb` 里同时运行两种语言的代码
5. **项目结构**：建立课程统一的文件夹约定，从第一次作业就做对

本章是**工具章**，每一节都有可以当场操作的步骤。
建议打开终端和 GitHub Desktop，跟着代码一步一步做一遍。
:::


---

## 0　引言：工具决定了你的分析能走多远

2019 年，《美国经济评论》要求所有投稿论文必须通过独立的「代码复现检验」——
编辑会把作者提供的代码和数据交给第三方，
如果无法复现表格和图表，论文不予接受。
这个要求让大量论文在审稿阶段就因为「代码在别的机器上跑不了」而被退回。

金融行业的要求更直接：一份回测报告，合规部门随时可能要求你
还原两年前那次策略调整的每一个细节——包括当时用的数据版本和代码逻辑。

本章要建立的工作流，解决的就是这个问题：
**让你的分析从第一天起就是可复现、可追溯、可分享的。**

贯穿本章的场景：把课程第一次个人作业（获取沪深 300 数据并分析）
做成一个完整的 GitHub 仓库项目，包含代码、数据说明和报告。


---

## 1　项目结构：从第一天就做对

好的项目结构让「三个月后的你」能在 5 分钟内找到任何文件。
本课程所有作业和最终项目使用统一的文件夹约定：

```
your_project/
├── README.md          ← 项目说明：问题、数据来源、运行方法、主要结论
├── requirements.txt   ← Python 依赖库清单
├── .gitignore         ← 告诉 Git 忽略哪些文件（数据文件、缓存等）
├── data_raw/          ← 原始数据，只读，永远不修改
├── data_clean/        ← 清洗后可用于分析的数据
├── data_temp/         ← 中间过程文件
├── codes/             ← 所有 .ipynb 和 .py 文件
└── output/            ← 图表、回归表等输出文件
```

运行下面的 code cell，一键在当前目录下创建这个结构，
同时生成 `README.md`、`requirements.txt` 和 `.gitignore` 模板。


In [ ]:
# ── 1.1  一键初始化项目结构 ─────────────────────────────────────────────
# 修改 PROJECT_NAME 为你的项目名，然后运行这个 cell。

import os
from pathlib import Path

PROJECT_NAME = 'hw01_hs300_analysis'   # ← 修改为你的项目名
ROOT = Path(PROJECT_NAME)

# ── 创建目录结构 ─────────────────────────────────────────────────────────
for d in ['data_raw', 'data_clean', 'data_temp', 'codes', 'output']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
    (ROOT / d / '.gitkeep').touch()    # 让空目录也能被 Git 追踪

# ── README.md 模板 ───────────────────────────────────────────────────────
(ROOT / 'README.md').write_text(
    f'# {PROJECT_NAME}\n\n'
    '## 研究问题\n\n（说明这个项目在分析什么问题）\n\n'
    '## 数据来源\n\n'
    '- **BasicInf**：CSMAR 基本信息库，获取方式：……\n'
    '- **FinRatio**：CSMAR 财务比率库，时间范围：2018–2023\n\n'
    '## 运行方法\n\n'
    '```bash\n'
    'conda activate dsfin\n'
    'pip install -r requirements.txt\n'
    'jupyter lab codes/analysis.ipynb\n'
    '```\n\n'
    '## 主要结论\n\n（分析完成后填写）\n',
    encoding='utf-8'
)

# ── requirements.txt ────────────────────────────────────────────────────
(ROOT / 'requirements.txt').write_text(
    'pandas>=2.0\nnumpy>=1.24\nmatplotlib>=3.7\n'
    'akshare>=1.0\nfredapi>=0.5\npyfixest>=0.18\n',
    encoding='utf-8'
)

# ── .gitignore ───────────────────────────────────────────────────────────
(ROOT / '.gitignore').write_text(
    '# Python 缓存\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n\n'
    '# 数据文件（版权限制，不上传 GitHub）\n'
    'data_raw/*.csv\ndata_raw/*.dta\ndata_raw/*.xlsx\ndata_raw/*.parquet\n\n'
    '# 环境文件\n.env\n*.egg-info/\n',
    encoding='utf-8'
)

# ── 输出结构树 ───────────────────────────────────────────────────────────
print(f'项目结构已创建：{ROOT.resolve()}\n')
for item in sorted(ROOT.rglob('*')):
    rel   = item.relative_to(ROOT)
    depth = len(rel.parts) - 1
    icon  = '📁 ' if item.is_dir() else '📄 '
    print('    ' * depth + icon + item.name)


::: {.callout-important}
## data_raw/ 只读原则

`data_raw/` 里的文件**永远不修改**。
所有清洗和处理的结果保存到 `data_clean/` 或 `data_temp/`。
这样即使处理逻辑出错，原始数据始终可以回溯。
:::

::: {.callout-tip}
## 提示词示范（生成 .gitignore）

````
我在做金融数据分析项目，数据来自 CSMAR（有版权限制，不能公开分发）。
请帮我生成一份完整的 .gitignore，要求：
1. 忽略 data_raw/ 下的所有数据文件（.csv .dta .xlsx .parquet）
2. 保留目录本身（用 .gitkeep 占位）
3. 忽略 Python 缓存和 Jupyter 检查点
4. 同时生成一段 README.md 说明，告诉他人如何自行获取和放置数据
````
:::


---

## 2　Git + GitHub：让代码可追溯

### 2.1　三个核心概念

| 概念 | 直觉比喻 | 说明 |
|------|----------|------|
| **Repository（仓库）**| 项目文件夹 | 包含所有代码、数据说明和历史记录 |
| **Commit（提交）** | 游戏存档点 | 把当前状态保存为一个快照，附上说明 |
| **Push / Pull** | 上传 / 下载 | 把本地仓库同步到 GitHub 云端 |

本课程只需掌握**主干工作流**：

```
写代码  →  Stage（暂存变更）  →  Commit（提交+说明）  →  Push（推送到 GitHub）
```

### 2.2　GitHub Desktop 操作步骤

GitHub Desktop 把命令行操作变成图形界面，推荐新手使用。

**初次设置（只做一次）：**
1. 下载：[https://desktop.github.com](https://desktop.github.com)
2. 安装后登录你的 GitHub 账号

**每次作业的标准流程：**
1. `File → New Repository` → 填写仓库名（如 `hw01_hs300`）和本地路径
2. 把刚才用 code cell 创建的项目文件夹移到这个仓库里
3. 写代码、生成图表
4. 在 GitHub Desktop 左侧看到变更的文件
5. 填写 **Summary**（commit 说明），点击 **Commit to main**
6. 点击 **Push origin** → 代码上传到 GitHub

::: {.callout-important}
## Commit 说明怎么写

好的说明让三个月后的你（和评分的老师）知道这次改了什么：

| 类型 | 示例 |
|------|------|
| `add` | `add: data cleaning script for FinRatio` |
| `fix` | `fix: stkcd format bug causing NaN after merge` |
| `docs` | `docs: update README with data sources` |
| `data` | `data: add HS300 daily returns 2022-2024` |

**不好的说明**：`update`、`fix bug`、`aaa`、`test`
:::


In [ ]:
# ── 2.3  用 Python 验证 Git 是否已安装 ───────────────────────────────────
import subprocess
import shutil

git_path = shutil.which('git')
if git_path:
    result = subprocess.run(['git', '--version'], capture_output=True, text=True)
    print(f'Git 已安装 ✓')
    print(f'路径：{git_path}')
    print(f'版本：{result.stdout.strip()}')
else:
    print('Git 未安装。')
    print('请从 https://git-scm.com 下载安装，然后重启终端和 Jupyter。')


### 2.4　命令行操作（可选）

如果你更习惯命令行，下面是等价的 Git 命令。
GitHub Desktop 和命令行操作的结果完全相同，选一种即可。

```bash
# 初始化仓库（只做一次）
git init
git remote add origin https://github.com/你的用户名/仓库名.git

# 日常提交流程
git add .                          # 暂存所有变更
git commit -m 'add: HS300 analysis'  # 提交并写说明
git push origin main               # 推送到 GitHub

# 查看历史记录
git log --oneline                  # 简洁版历史
git diff HEAD~1 HEAD               # 与上一个版本的差异
```

::: {.callout-tip}
## 提示词示范（Git 操作问题）

````
我在用 Git 时遇到了以下问题：[描述现象或粘贴错误信息]
我的操作步骤是：[依次列出你做了什么]
当前仓库状态：[粘贴 git status 的输出]

请帮我：
1. 解释这个问题的原因
2. 给出恢复到正常状态的步骤（不会丢失我的代码）
3. 说明如何避免这个问题再次发生
````
:::


---

## 3　Marp：用 Markdown 做幻灯片

**Marp** 让你用纯文本写幻灯片，无需打开 PowerPoint。
对于数据分析汇报来说，它有一个关键优势：
代码和图表可以直接嵌入，内容更新时幻灯片自动同步。

### 3.1　安装

在 VS Code 里安装扩展：搜索 `Marp for VS Code`，点击安装。
（不需要安装任何命令行工具，扩展内置了渲染引擎。）

### 3.2　最小可运行示例

新建一个 `.md` 文件（如 `slides.md`），粘贴下面的内容：

````markdown
---
marp: true
theme: default
paginate: true
---

# 沪深 300 收益率分析

金融数据分析与建模 · 第一次作业

---

## 数据概览

- 数据来源：AKShare
- 时间范围：2022–2024
- 观测天数：约 730 个交易日

---

## 收益率分布

![w:700](output/ch01_ret_dist.png)

---

## 主要结论

1. 日均收益率：+0.03%（年化约 7%）
2. 波动率：日度 1.2%，年化约 19%
3. 最大回撤：-32%（2022 年 4 月）
````

### 3.3　导出幻灯片

在 VS Code 里按 `Ctrl+Shift+P`，搜索 `Marp: Export Slide Deck`：
- 选 **PDF**：打印质量，可在任何设备打开
- 选 **HTML**：交互式，可在浏览器里播放
- 选 **PPTX**：可在 PowerPoint 里进一步编辑

::: {.callout-note}
## 常用 Marp 格式技巧

```markdown
![w:600](图片路径)        # 设置图片宽度为 600px
![bg right:40%](图片)    # 图片作为右侧 40% 的背景
<!-- _class: lead -->    # 使用「居中大字」主题样式（当前页）
$$E = mc^2$$             # LaTeX 数学公式
```
:::

::: {.callout-tip}
## 提示词示范（Marp 幻灯片）

````
我需要用 Marp 做一份 5 页的数据分析汇报幻灯片，内容如下：
- 第 1 页：标题页，课程名称、作业编号、姓名
- 第 2 页：研究问题（一句话描述 + 3 个要点）
- 第 3 页：数据说明（表格展示数据来源、时间范围、变量）
- 第 4 页：主要图表（插入 output/main_plot.png，附说明文字）
- 第 5 页：结论（3 条结论 + 局限性说明）

主题使用 default，启用页码，配色要简洁专业。
请生成完整的 .md 文件内容。
````
:::


---

## 4　Quarto：把 Notebook 变成报告和网站

**Quarto** 是一个科学出版系统，能把 `.ipynb` 或 `.qmd` 文件
渲染成 HTML 网页、PDF、Word 文档，或整本电子书（GitHub Pages）。

本课程用 Quarto 做两件事：
1. **课程项目报告**：把分析结果渲染成一份精美的 HTML 报告
2. **可选加分项**：发布为 GitHub Pages，让报告可以通过网址访问

### 4.1　安装

从 [https://quarto.org](https://quarto.org) 下载并安装。
安装完成后在终端验证：

```bash
quarto --version   # 应输出版本号，如 1.5.x
```

### 4.2　核心概念：`.qmd` 文件

`.qmd`（Quarto Markdown）= Markdown 文本 + 可执行代码块 + YAML 元数据。
它和 Jupyter Notebook 的区别是：`.qmd` 是纯文本，可以用 Git 追踪每个字的变化。

### 4.3　渲染命令

```bash
quarto render report.qmd              # 生成 HTML（默认）
quarto render report.qmd --to pdf     # 生成 PDF（需要 TinyTeX）
quarto preview report.qmd             # 实时预览（保存即刷新）
```

> **参考示例**：本课程讲义就是用 Quarto Book + GitHub Pages 发布的。
> 你的项目报告可以做成同样的形式：
> [https://lianxhcn.github.io/quarto_book/](https://lianxhcn.github.io/quarto_book/)


In [ ]:
# ── 4.4  生成课程项目报告模板（report.qmd）─────────────────────────────
from pathlib import Path

template = '''---
title: "课程项目报告"
subtitle: "金融数据分析与建模"
author: "姓名 | 学号"
date: today
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: true       # 代码默认折叠，点击展开
    code-tools: true      # 右上角显示「查看代码」按钮
    theme: cosmo
    self-contained: true  # 打包成单个 HTML 文件
    fig-width: 8
    fig-dpi: 150
execute:
  warning: false
  message: false
jupyter: dsfin
---

## 1  研究背景与问题

（说明你分析的问题是什么，为什么值得研究，引用 1-2 篇相关文献。）

## 2  数据来源与处理

```{{python}}
#| label: data-loading
#| fig-cap: "数据读取与基本描述"
import pandas as pd
# df = pd.read_csv('data_clean/merged_clean.csv')
# df.describe()
```

## 3  方法

（说明使用的分析方法，引用理论依据。）

## 4  结果与分析

```{{python}}
#| label: main-results
# 主要分析代码
```

## 5  结论

## 参考文献
'''

path = Path('report.qmd')
path.write_text(template, encoding='utf-8')
print(f'report.qmd 已生成：{path.resolve()}')
print()
print('常用渲染命令（在终端里运行）：')
print('  quarto render report.qmd          # 生成 HTML')
print('  quarto render report.qmd --to pdf  # 生成 PDF')
print('  quarto preview report.qmd          # 实时预览（推荐写作时使用）')


### 4.5　发布为 GitHub Pages（可选加分项）

1. 在仓库里新建 `_quarto.yml`，内容：
   ```yaml
   project:
     type: website
     output-dir: docs
   ```
2. 运行 `quarto render`，生成 `docs/` 目录
3. 在 GitHub 仓库设置里，找到 `Pages`，
   `Source` 选 `Deploy from a branch`，
   Branch 选 `main`，目录选 `/docs`
4. 几分钟后，报告就可以通过 `https://你的用户名.github.io/仓库名/` 访问了

::: {.callout-tip}
## 提示词示范（Quarto 报告）

````
我想用 Quarto 把我的 Jupyter Notebook 分析结果做成一份 HTML 报告。
要求：
- 目录自动生成，深度到 3 级标题
- 代码默认折叠，读者可以点击展开
- 图表宽度 8 英寸，分辨率 150 dpi
- 打包成单个 HTML 文件（方便邮件发送）
- kernel 使用 dsfin

请帮我生成完整的 YAML 头部（front matter）。
````
:::


---

## 5　Stata × Python 双语工作流

本课程的回归分析章节（T3–T6）同时使用 Python 和 Stata。
两者不是替代关系，而是**分工协作**：

| 场景 | 推荐工具 | 理由 |
|------|----------|------|
| 数据获取 / 清洗 / 可视化 | **Python** | 生态更丰富，API 更多 |
| OLS / FE / HDFE 回归 | **Stata 或 Python** | `reghdfe`+`esttab` 是实证研究标准；`pyfixest` 结果完全一致 |
| DID / RDD / PSM | **Stata 或 Python** | Stata 的 `csdid`、`rdrobust` 更成熟；Python 有同名移植包 |
| 机器学习 | **Python** | Stata 的 ML 支持极为有限 |

工作流的标准链路：

```
Python 获取 + 清洗数据  →  导出 .dta  →  Stata 做回归  →  结果回 Python 可视化
                         （或直接用 pyfixest，无需 Stata）
```

### 5.1　方案一：在 Jupyter 里直接运行 Stata（需要 Stata 17+）

Stata 17+ 提供了 `pystata` 库，让你在 code cell 里用 `%%stata` magic 写 Stata 代码，
就像写普通 Python 一样。

::: {.callout-note}
## 前提条件

- 已安装 **Stata 17 或以上版本**
- `pystata` 库已包含在 Stata 安装目录里（无需额外安装）
- 需要在 Python 里指定 Stata 的安装路径
:::


In [ ]:
# ── 5.2  pystata 配置与测试 ─────────────────────────────────────────────
# 注意：这段代码只有安装了 Stata 17+ 的机器上才能运行。
# 没有 Stata 的同学跳过本节，使用 5.4 节的 pyfixest 方案。

import importlib.util

if importlib.util.find_spec('pystata') is not None:
    import pystata

    # ── 根据你的操作系统修改路径 ─────────────────────────────────────────
    # Windows：pystata.config.init('mp', r'C:\Program Files\Stata18')
    # macOS：  pystata.config.init('mp', '/Applications/Stata/')
    # Linux：  pystata.config.init('mp', '/usr/local/stata18')
    # pystata.config.init('mp', '/你的/Stata/路径')

    print('pystata 可用，Stata 双语模式已就绪 ✓')
    print('在 code cell 开头加 %%stata 即可写 Stata 代码')
else:
    print('pystata 不可用（未安装 Stata 或路径未配置）。')
    print('请使用 5.4 节的 pyfixest 方案，结果与 Stata 完全一致。')


### 5.3　双语工作流示例

以下是一个完整的「Python 准备数据 → Stata 回归 → Python 可视化」示例，
适合在已配置 `pystata` 的机器上运行：

**cell 1：Python 准备数据**
```python
import pandas as pd
df = pd.read_csv('data_clean/merged_clean.csv')
df.to_stata('data_temp/for_stata.dta', write_index=False)
print('数据已导出至 data_temp/for_stata.dta')
```

**cell 2：Stata 回归**
```stata
%%stata
* ── 高维固定效应回归 ──────────────────────────────────────
* 需要提前安装：ssc install reghdfe, replace
*               ssc install ftools, replace
use "data_temp/for_stata.dta", clear

reghdfe ROA_w soe Size_w Leverage_w, ///
    absorb(stkcd year) vce(cluster stkcd)

* 输出规范回归表（ssc install estout, replace）
esttab, star(* 0.1 ** 0.05 *** 0.01) se b(%9.4f) r2 scalars(N)
```

**cell 3：Python 绘制系数图**
```python
# Stata 结果已在 cell 输出里显示。
# 如需进一步可视化，可用 pystata 的 return 接口获取回归对象，
# 或手动整理系数后用 matplotlib 绘制。
```

### 5.4　方案二：pyfixest（无 Stata 时的完整替代）

`pyfixest` 是最接近 Stata `reghdfe` 的 Python 固定效应回归库，
语法几乎一一对应，输出格式也和 `esttab` 类似。
**没有 Stata 的同学使用这个方案，结果与 Stata 精确一致。**


In [ ]:
# ── 5.5  pyfixest 安装验证 + 语法对照演示 ──────────────────────────────

try:
    import pyfixest as pf
    print(f'pyfixest {pf.__version__} ✓')
except ImportError:
    print('请先安装：pip install pyfixest')
    raise

import numpy as np
import pandas as pd

# ── 构造演示面板数据 ──────────────────────────────────────────────────────
rng = np.random.default_rng(42)
n_firms, n_years = 50, 6
n = n_firms * n_years

demo = pd.DataFrame({
    'firm' : np.repeat(np.arange(1, n_firms+1), n_years),
    'year' : np.tile(np.arange(2018, 2018+n_years), n_firms),
    'ROA'  : rng.normal(0.05, 0.03, n),
    'Size' : rng.normal(23, 1.5, n),
    'Lev'  : rng.beta(2, 5, n),
    'soe'  : rng.integers(0, 2, n),
})

# ── 三个模型：无 FE → 单向 FE → 双向 FE ────────────────────────────────
m1 = pf.feols('ROA ~ soe + Size + Lev',
              data=demo, vcov='HC1')                    # OLS，异方差稳健 SE

m2 = pf.feols('ROA ~ soe + Size + Lev | firm',
              data=demo, vcov={'CRV1': 'firm'})         # 公司 FE + 聚类 SE

m3 = pf.feols('ROA ~ soe + Size + Lev | firm + year',
              data=demo, vcov={'CRV1': 'firm'})         # 双向 FE + 聚类 SE

# ── 并排输出回归表（等价于 Stata esttab）────────────────────────────────
pf.etable([m1, m2, m3],
          coef_fmt='b (se)',
          digits=4,
          stars=True)


::: {.callout-note}
## pyfixest 与 Stata 语法对照表

| 操作 | Stata | pyfixest |
|------|-------|----------|
| OLS | `reg y x1 x2` | `feols('y ~ x1 + x2', data=df)` |
| 公司 FE | `reghdfe y x1, absorb(firm)` | `feols('y ~ x1 \| firm', data=df)` |
| 双向 FE | `reghdfe y x1, absorb(firm year)` | `feols('y ~ x1 \| firm + year', data=df)` |
| 异方差稳健 SE | `vce(robust)` | `vcov='HC1'` |
| 聚类 SE（公司）| `vce(cluster firm)` | `vcov={'CRV1': 'firm'}` |
| 输出回归表 | `esttab m1 m2 m3` | `pf.etable([m1, m2, m3])` |

**结果验证**：用同一份数据，Stata `reghdfe` 和 `pyfixest feols` 的系数、
标准误应精确到小数点后 4 位相同。
如有差异，检查固定效应维度是否完全一致。
:::

::: {.callout-tip}
## 提示词模板：Stata ↔ pyfixest 互译

这个提示词具有普适性——整个课程的回归部分都可以套用：

````
我有以下 Stata 回归代码，请帮我转写为等价的 pyfixest Python 代码：

[粘贴 Stata 代码]

DataFrame 名为 df，变量名与 Stata 代码一致。
要求：
1. 固定效应设置与 Stata absorb() 完全一致
2. 聚类标准误设置与 Stata vce(cluster) 完全一致
3. 用 pf.etable() 输出回归表，格式与 esttab 类似
4. 说明如何验证两者结果一致（应精确到小数点后 4 位）
````
:::


---

## 6　综合演示：一次作业的完整工作流

把本章所有工具串在一起，展示课程第一次作业从零到提交的全流程。
这个流程在后续每次作业和最终项目中都会重复。

```
① GitHub 建仓库（网页操作，约 2 分钟）
   └─ 仓库名：hw01_hs300_analysis
   └─ 勾选「Add a README file」

② 本地初始化项目结构（运行第 1 节的 code cell）
   └─ 修改 PROJECT_NAME = 'hw01_hs300_analysis'
   └─ 把生成的文件夹克隆到 GitHub Desktop 里的仓库目录

③ 写分析代码（见 03_api_data.ipynb）
   └─ 获取沪深 300 日度收益率
   └─ 计算描述统计和年化指标
   └─ 生成图表，保存到 output/

④ 用 Marp 制作 slides（slides.md，约 10 分钟）
   └─ 5 页：标题 + 数据 + 图表 + 统计 + 结论
   └─ 导出为 PDF

⑤ 用 Quarto 生成 HTML 报告（report.qmd）
   └─ quarto render report.qmd
   └─ 在浏览器里检查效果

⑥ Commit + Push
   └─ Summary：'add: hw01 HS300 analysis complete'
   └─ 在 GitHub 上检查仓库内容

⑦ （可选）发布 GitHub Pages
   └─ 仓库 Settings → Pages → Deploy from /docs
   └─ 等待 1-2 分钟，报告上线
```


In [ ]:
# ── 6.1  快速健康检查：本章工具是否全部就绪 ─────────────────────────────
import subprocess
import shutil
import importlib

checks = []

# Git
git_ok = shutil.which('git') is not None
checks.append(('Git', git_ok, 'https://git-scm.com' if not git_ok else ''))

# Quarto
quarto_ok = shutil.which('quarto') is not None
checks.append(('Quarto', quarto_ok, 'https://quarto.org' if not quarto_ok else ''))

# pyfixest
pyfixest_ok = importlib.util.find_spec('pyfixest') is not None
checks.append(('pyfixest', pyfixest_ok,
               'pip install pyfixest' if not pyfixest_ok else ''))

# pystata（可选）
stata_ok = importlib.util.find_spec('pystata') is not None
checks.append(('pystata (可选)', stata_ok,
               '需要安装 Stata 17+' if not stata_ok else ''))

print(f'{'工具':<20} {'状态':<8} 说明')
print('─' * 55)
for name, ok, note in checks:
    flag = '✓' if ok else '✗'
    print(f'{name:<20} {flag:<8} {note}')

print()
must_ok = all(ok for name, ok, _ in checks if '可选' not in name)
if must_ok:
    print('✅  必要工具全部就绪，可以开始写作业了。')
else:
    print('⚠️  请先安装缺失的工具，再继续后续章节。')


---

## 7　章末练习

**练习 1（操作，必做）**
按本章第 1 节和第 2 节的步骤：
① 在 GitHub 上建立名为 `hw01_hs300` 的仓库；
② 运行第 1 节的 code cell 初始化项目结构；
③ 做一次 Commit + Push，提交内容：README.md + requirements.txt + .gitignore；
④ 截图 GitHub 仓库主页，提交到作业系统。

**练习 2（工具，必做）**
用 Marp 制作一份 5 页幻灯片（主题、数据说明、图表×2、结论），
导出为 PDF，放入仓库的 `output/` 目录并 Push。

**练习 3（可选加分）**
用第 4.5 节的步骤把报告发布为 GitHub Pages，
在 README.md 里添加网站链接，Push 后把链接提交到作业系统。

**练习 4（思考题）**
设想你的队友要复现你的分析，但没有 CSMAR 账号。
你的 README.md 里应该写哪些内容，让他可以自行获取数据并运行代码？
写出你认为完整的 README「数据来源」一节的内容。
